# Device Fingerprint Probability Plotting

This notebook contains the hardcoded class probabilities for `192.168.10.20` and `192.168.10.21`, along with `plot_device_fingerprints` and `export_graph_data` functions.

In [37]:
import os
import ast
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Global constants
DEVICE_COL = 'device_ip'

def log(msg):
    print(f"[LOG] {msg}")

def format_class_label(label):
    return str(label).upper()

In [38]:
# Hardcoded Device Probabilities

# 192.168.10.20
dev_20_probs = {
    'recon': 0.613266580221998,
    'benign': 0.2466204008056395,
    'malware': 0.0482044306930693,
    'dos': 0.044734499999999996,
    'ddos': 0.04395977720207254,
    'web': 0.043212178929765886
}

# 192.168.10.21
dev_21_probs = {
    'benign': 0.3268161526104418,
    'malware': 0.17641020000000002,
    'ddos': 0.16239210351758795,
    'recon': 0.16165303024193547
}

# Constructing sample DataFrames matching expected structure
df_meta = pd.DataFrame({
    DEVICE_COL: ['192.168.10.20', '192.168.10.21'],
    'container_name': ['device_192_168_10_20', 'device_192_168_10_21']
})

fingerprint_df = pd.DataFrame({
    'predicted_label': ['recon', 'benign'],
    'confidence': [0.613266580221998, 0.3268161526104418],
    'is_attack': [True, False],
    'fingerprint_hash': ['hash_10_20', 'hash_10_21'],
    'prob_vector': [[0.613266580221998, 0.2466204008056395, 0.0482044306930693, 0.044734499999999996, 0.04395977720207254, 0.043212178929765886], [0.3268161526104418, 0.17641020000000002, 0.16239210351758795, 0.16165303024193547]],
    'top_classes': [dev_20_probs, dev_21_probs],
    'window_start': [1000, 1001]
})
fingerprint_df.attrs['class_names'] = ['recon', 'benign', 'malware', 'dos', 'ddos', 'web']

X = pd.DataFrame({
    'network_bytes_in': [1024, 2048],
    'network_bytes_out': [512, 1024],
    'log_packet_count': [50, 100],
    'network_tcp_flags': [2, 2],
    'log_duration': [10, 20]
})

class MockBooster:
    def get_score(self, importance_type='gain'):
        return {
            'network_bytes_in': 120.5,
            'network_bytes_out': 98.2,
            'log_packet_count': 85.0,
            'network_tcp_flags': 45.1,
            'log_duration': 30.2
        }

class MockModel:
    def get_booster(self):
        return MockBooster()

model = MockModel()

In [ ]:
def plot_device_probability_distribution(df_meta, fingerprint_df, output_path="device_probability_distributions.png"):
    import ast
    import pandas as pd
    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt

    # Set notebook styling parameters
    plt.rcParams.update({
        "figure.facecolor":  "#ffffff",
        "axes.facecolor":    "#ffffff",
        "axes.edgecolor":    "#444444",
        "axes.labelcolor":   "#000000",
        "axes.titlecolor":   "#000000",
        "xtick.color":       "#000000",
        "ytick.color":       "#000000",
        "text.color":        "#000000",
        "grid.color":        "#d0d0d0",
        "grid.linestyle":    "--",
        "grid.alpha":        0.5,
        "font.family":       "monospace",
    })

    ACCENT_BENIGN = "#3fb950"  # Green
    ACCENT_ATTACK = "#f85149"  # Red

    # Combine metadata and fingerprint records
    combined = pd.concat(
        [df_meta.reset_index(drop=True), fingerprint_df.reset_index(drop=True)],
        axis=1
    )

    devices = combined[DEVICE_COL].dropna().unique().tolist()
    n_devices = len(devices)

    if n_devices == 0:
        print("[LOG] No devices found for plotting.")
        return

    fig, axes = plt.subplots(
        nrows=n_devices, 
        ncols=1, 
        figsize=(10, 4 * n_devices), 
        squeeze=False
    )

    for dev_idx, device in enumerate(devices):
        ax = axes[dev_idx, 0]
        dev_rows = combined[combined[DEVICE_COL] == device]
        container_name = dev_rows['container_name'].iloc[0] if 'container_name' in dev_rows.columns else device

        # Aggregate class probabilities across windows
        top_class_probs = {}
        for _, row in dev_rows.iterrows():
            tc = row['top_classes'] if isinstance(row['top_classes'], dict) else ast.literal_eval(str(row['top_classes']))
            for cls, prob in tc.items():
                top_class_probs[cls] = top_class_probs.get(cls, 0) + prob

        total_weight = sum(top_class_probs.values())
        if total_weight > 0:
            top_class_probs = {
                k: v / total_weight
                for k, v in sorted(top_class_probs.items(), key=lambda x: x[1], reverse=True)
            }

        cls_labels = list(top_class_probs.keys())
        cls_vals = list(top_class_probs.values())

        # Print probability values to stdout
        print(f"========================================")
        print(f"Probability Distribution: {container_name} ({device})")
        print(f"========================================")
        for cls, prob in top_class_probs.items():
            print(f"{format_class_label(cls):<12}: {prob:.4f} ({prob:.2%})")
        print()

        # Bar chart colours based on class
        cls_colours = [
            ACCENT_BENIGN if cls.lower() == "benign" else ACCENT_ATTACK 
            for cls in cls_labels
        ]

        bars = ax.barh(
            range(len(cls_labels)),
            cls_vals,
            color=cls_colours,
            height=0.6,
            edgecolor="none",
            alpha=0.85
        )

        # Annotate bars with percentage values
        for i, val in enumerate(cls_vals):
            ax.text(
                val + 0.005, i,
                f"{val:.1%}", va="center", ha="left",
                fontsize=8, color="black"
            )

        ax.set_yticks(range(len(cls_labels)))
        ax.set_yticklabels([format_class_label(l) for l in cls_labels], fontsize=8)
        ax.set_xlabel("Mean probability across windows", fontsize=8)
        ax.set_title(f"Device: {container_name} ({device})", fontsize=10, fontweight="bold", pad=10)
        ax.set_xlim(0, 1.2)
        ax.invert_yaxis()
        ax.grid(axis="x")

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="#ffffff")
    print(output_path)
    plt.show()
    plt.close()
    print(f"[LOG] Device probability plot saved to {output_path}")

In [54]:
plot_device_probability_distribution(
    df_meta=df_meta, 
    fingerprint_df=fingerprint_df, 
    output_path="device_fingerprints.png"
)

Probability Distribution: device_192_168_10_20 (192.168.10.20)
RECON       : 0.5897 (58.97%)
BENIGN      : 0.2371 (23.71%)
MALWARE     : 0.0464 (4.64%)
DOS         : 0.0430 (4.30%)
DDOS        : 0.0423 (4.23%)
WEB         : 0.0416 (4.16%)

Probability Distribution: device_192_168_10_21 (192.168.10.21)
BENIGN      : 0.3951 (39.51%)
MALWARE     : 0.2132 (21.32%)
DDOS        : 0.1963 (19.63%)
RECON       : 0.1954 (19.54%)

Saving to: /content/device_fingerprints.png
/content/device_fingerprints.png
[LOG] Device probability plot saved to /content/device_fingerprints.png
